# EMA Cross Backtest — composite (aggregated) bar type

Self-contained NautilusTrader backtest of the **EMA Cross** strategy.

- The strategy in [ema_cross.py](ema_cross.py) is **not modified**. A timeframe-aware
  variant of it is defined **inside this notebook only** (per the task).
- Strategy config takes a **timeframe** (`5min`, `15min`, `30min`, `45min`, `1hr`).
  Here we pass `15min`.
- The strategy subscribes to a **composite bar type**:
  `EURUSD.FOREX_MS-15-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL`.
  We load **1-minute** data; the engine's internal aggregator converts it to the
  configured timeframe on the fly. No pre-aggregated catalog is needed.
- Both **ASK and BID** 1-minute data are loaded. The simulated matching engine
  needs both sides to fill orders — with only one side every order is rejected
  with a *market not found* error.
- After the run we generate the **order fill**, **position** and **account** reports.

## 1. Imports & project path

In [1]:
import sys
from pathlib import Path
from decimal import Decimal

import pandas as pd

# Put the project root on sys.path so `core.*` imports resolve whether the
# notebook is launched from `strategies/` or from the project root.
_nb_dir = Path.cwd()
PROJECT_ROOT = _nb_dir if (_nb_dir / "core").exists() else _nb_dir.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.csv_loader import concat_side
from core.instrument_factory import create_instrument
from core.nautilus_loader import wrangle_bars

from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import (
    BacktestEngineConfig,
    LoggingConfig,
    PositiveInt,
    RiskEngineConfig,
    StrategyConfig,
)
from nautilus_trader.core.correctness import PyCondition
from nautilus_trader.indicators import ExponentialMovingAverage
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
from nautilus_trader.model.identifiers import InstrumentId, TraderId, Venue
from nautilus_trader.model.instruments import Instrument
from nautilus_trader.model.objects import Money
from nautilus_trader.trading.strategy import Strategy

print(f"Project root: {PROJECT_ROOT}")

Project root: d:\mcube_test_version_1_updated\m-cube_version1


## 2. Configuration

`TIMEFRAME` is the only knob you need to change to test another timeframe.
`TIMEFRAME_MAP` translates the friendly label into a NautilusTrader bar
aggregation spec used to build the composite bar type.

In [3]:
# Friendly label -> NautilusTrader bar aggregation spec
TIMEFRAME_MAP = {
    "5min":  "5-MINUTE",
    "15min": "15-MINUTE",
    "30min": "30-MINUTE",
    "45min": "45-MINUTE",
    "1hr":   "1-HOUR",
}

# --- backtest configuration --------------------------------------------------
DATA_DIR = Path(r"D:\MS\Dataset\Commodities_yyyy\Commodities\LIGHTCMDUSD\2020\04")  # April 2024, day-wise
VENUE = "Commodities_MS"
TIMEFRAME = "15min"          # one of: 5min, 15min, 30min, 45min, 1hr
FAST_EMA = 10
SLOW_EMA = 20
TRADE_SIZE = 100_000         # units of EUR per order (1 standard lot)
STARTING_CAPITAL = 1_000_000 # USD
# -----------------------------------------------------------------------------

assert TIMEFRAME in TIMEFRAME_MAP, f"TIMEFRAME must be one of {list(TIMEFRAME_MAP)}"
AGG_SPEC = TIMEFRAME_MAP[TIMEFRAME]
print(f"Data dir : {DATA_DIR}")
print(f"Timeframe: {TIMEFRAME} -> {AGG_SPEC}")

Data dir : D:\MS\Dataset\Commodities_yyyy\Commodities\LIGHTCMDUSD\2020\04
Timeframe: 15min -> 15-MINUTE


## 3. Load 1-minute ASK & BID CSV data

Each day folder holds two files — `DD.MM.YYYY_ASK_OHLCV.csv` and
`DD.MM.YYYY_BID_OHLCV.csv` — both 1-minute bars. We concatenate every day
for each side. `concat_side` reads the files in parallel, parses the
`DD.MM.YYYY HH:MM:SS GMT±HHMM` timestamps to UTC, sorts and de-duplicates.

In [4]:
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

ask_files = sorted(str(p) for p in DATA_DIR.rglob("*_ASK_OHLCV.csv"))
bid_files = sorted(str(p) for p in DATA_DIR.rglob("*_BID_OHLCV.csv"))
print(f"ASK files found: {len(ask_files)}")
print(f"BID files found: {len(bid_files)}")
assert ask_files, "No *_ASK_OHLCV.csv files found under DATA_DIR"
assert bid_files, "No *_BID_OHLCV.csv files found under DATA_DIR"

# The FX daily files use a 'timestamp' column (not the crypto default 'ts').
ask_df = concat_side(ask_files, timestamp_column="timestamp")
bid_df = concat_side(bid_files, timestamp_column="timestamp")

print(f"\nASK rows: {len(ask_df):,}  ({ask_df.index[0]}  ->  {ask_df.index[-1]})")
print(f"BID rows: {len(bid_df):,}  ({bid_df.index[0]}  ->  {bid_df.index[-1]})")
ask_df.head()

ASK files found: 25
BID files found: 25

ASK rows: 28,659  (2020-04-01 00:00:00+00:00  ->  2020-04-30 23:59:00+00:00)
BID rows: 28,677  (2020-04-01 00:00:00+00:00  ->  2020-04-30 23:59:00+00:00)


,open,high,low,close,volume
timestamp,,,,,
2020-04-01 00:00:00+00:00,20.335,20.355,20.305,20.325,0.0
2020-04-01 00:01:00+00:00,20.335,20.335,20.305,20.315,0.0
2020-04-01 00:02:00+00:00,20.325,20.435,20.325,20.425,0.0
2020-04-01 00:03:00+00:00,20.430,20.490,20.405,20.465,0.0
2020-04-01 00:04:00+00:00,20.470,20.516,20.445,20.515,0.0


## 4. Create the instrument & wrangle 1-minute bars

`EURUSD` on venue `FOREX_MS` — the venue must match the `InstrumentId` so
the engine's `add_venue` and the bar types line up. We wrangle the ASK and
BID frames into 1-minute `EXTERNAL` bars; these are the bars the engine
actually consumes.

In [5]:
instrument = create_instrument("EUR", "USD", venue=VENUE)
print(f"Instrument     : {instrument.id}")
print(f"Price precision: {instrument.price_precision}")
print(f"Size precision : {instrument.size_precision}")

# 1-minute EXTERNAL bars. price_type ASK/BID -> bar types:
#   EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL
#   EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL
ask_bars = wrangle_bars(ask_df, instrument, timeframe="1-MINUTE", price_type="ASK")
bid_bars = wrangle_bars(bid_df, instrument, timeframe="1-MINUTE", price_type="BID")

print(f"\nASK 1-min bars: {len(ask_bars):,}  type={ask_bars[0].bar_type}")
print(f"BID 1-min bars: {len(bid_bars):,}  type={bid_bars[0].bar_type}")

Instrument     : EURUSD.Commodities_MS
Price precision: 5
Size precision : 0

ASK 1-min bars: 28,659  type=EURUSD.Commodities_MS-1-MINUTE-ASK-EXTERNAL
BID 1-min bars: 28,677  type=EURUSD.Commodities_MS-1-MINUTE-BID-EXTERNAL


## 5. Strategy — EMA Cross with a timeframe parameter

This is a copy of the logic in [ema_cross.py](ema_cross.py), modified **here
only** so the config takes a `timeframe`. The strategy builds the composite
bar type from it and subscribes:

```
EURUSD.FOREX_MS-15-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL
```

The `@1-MINUTE-EXTERNAL` suffix tells the engine to feed the loaded
1-minute BID bars into an internal aggregator that emits 15-minute bars.
Indicators are updated directly via `handle_bar` so the strategy is robust
to the composite-vs-standard bar-type key.

In [ ]:
class EMACrossTFConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    timeframe: str = "15-MINUTE"          # NautilusTrader aggregation spec
    trade_size: Decimal = Decimal("100000")
    fast_ema_period: PositiveInt = 10
    slow_ema_period: PositiveInt = 20


class EMACrossTFStrategy(Strategy):
    """Buy when fast EMA >= slow EMA, sell on cross below.

    Trades the configured timeframe by subscribing to a composite bar type
    so the engine aggregates the loaded 1-minute BID bars in-process.
    """

    def __init__(self, config: EMACrossTFConfig) -> None:
        PyCondition.is_true(
            config.fast_ema_period < config.slow_ema_period,
            f"fast_ema_period ({config.fast_ema_period}) must be < "
            f"slow_ema_period ({config.slow_ema_period})",
        )
        super().__init__(config)
        self.instrument: Instrument = None
        self.bar_type: BarType = None
        self.fast_ema = ExponentialMovingAverage(config.fast_ema_period)
        self.slow_ema = ExponentialMovingAverage(config.slow_ema_period)

    def on_start(self) -> None:
        self.instrument = self.cache.instrument(self.config.instrument_id)
        if self.instrument is None:
            self.log.error(f"Could not find instrument {self.config.instrument_id}")
            self.stop()
            return
        # Composite bar type: aggregate 1-MINUTE-EXTERNAL BID bars into the
        # configured timeframe INTERNAL bars.
        self.bar_type = BarType.from_str(
            f"{self.config.instrument_id}-{self.config.timeframe}-BID-INTERNAL"
            f"@1-MINUTE-EXTERNAL"
        )
        self.log.info(f"Subscribing to composite bar type: {self.bar_type}")
        self.subscribe_bars(self.bar_type)

    def on_bar(self, bar: Bar) -> None:
        # Feed indicators directly from the aggregated bar.
        self.fast_ema.handle_bar(bar)
        self.slow_ema.handle_bar(bar)
        if not (self.fast_ema.initialized and self.slow_ema.initialized):
            return

        iid = self.config.instrument_id
        if self.fast_ema.value >= self.slow_ema.value:
            if self.portfolio.is_flat(iid):
                self._submit_order(OrderSide.BUY)
            elif self.portfolio.is_net_short(iid):
                self.close_all_positions(iid)
                self._submit_order(OrderSide.BUY)
        else:
            if self.portfolio.is_flat(iid):
                self._submit_order(OrderSide.SELL)
            elif self.portfolio.is_net_long(iid):
                self.close_all_positions(iid)
                self._submit_order(OrderSide.SELL)

    def _submit_order(self, side: OrderSide) -> None:     
        fp = int(self.config.fast_ema_period)
        sp = int(self.config.slow_ema_period)
        fv, sv = self.fast_ema.value, self.slow_ema.value
        if side == OrderSide.BUY:
            reason = f"EMA Cross BUY: fast({fp})={fv:.5f} >= slow({sp})={sv:.5f}"
        else:
            reason = f"EMA Cross SELL: fast({fp})={fv:.5f} < slow({sp})={sv:.5f}"
        order = self.order_factory.market(
            instrument_id=self.config.instrument_id,
            order_side=side,
            quantity=self.instrument.make_qty(self.config.trade_size),
            time_in_force=TimeInForce.GTC,
            tags=[reason],
        )
        self.submit_order(order)

    def on_stop(self) -> None:
        self.cancel_all_orders(self.config.instrument_id)
        self.close_all_positions(self.config.instrument_id)


print("EMACrossTFStrategy defined.")

EMACrossTFStrategy defined.


## 6. Build the backtest engine

Both ASK and BID 1-minute bars are added as data:

- The **BID** bars feed the strategy's composite aggregator (15-minute bars)
  **and** are used to fill SELL orders.
- The **ASK** bars are used by the matching engine to fill BUY orders.

If only one side were loaded, every order would be rejected (*market not
found*) because the matching engine can't price the missing side.

In [7]:
engine = BacktestEngine(config=BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-001"),
    logging=LoggingConfig(bypass_logging=True),
    risk_engine=RiskEngineConfig(bypass=True),
))

engine.add_venue(
    venue=Venue(VENUE),
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=USD,
    starting_balances=[Money(STARTING_CAPITAL, USD)],
    default_leverage=Decimal(1),
)

engine.add_instrument(instrument)
engine.add_data(ask_bars)   # ASK 1-min -> matching engine (BUY fills)
engine.add_data(bid_bars)   # BID 1-min -> composite aggregator + SELL fills
print("Engine configured: venue + instrument + ASK/BID 1-minute data added.")

Engine configured: venue + instrument + ASK/BID 1-minute data added.


## 7. Add the strategy & run the backtest

The timeframe is passed through the strategy **config** — change `TIMEFRAME`
in section 2 to backtest 5/30/45-minute or 1-hour without touching anything
else.

In [8]:
strategy_config = EMACrossTFConfig(
    instrument_id=instrument.id,
    timeframe=AGG_SPEC,                       # e.g. "15-MINUTE"
    trade_size=Decimal(str(TRADE_SIZE)),
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
)
strategy = EMACrossTFStrategy(strategy_config)
engine.add_strategy(strategy)
print(f"Strategy added: timeframe={AGG_SPEC}, fast_ema={FAST_EMA}, slow_ema={SLOW_EMA}")

engine.run()
print("\nBacktest finished.")

Strategy added: timeframe=15-MINUTE, fast_ema=10, slow_ema=20

Backtest finished.


## 8. Reports

The task asked for `engine.generate_order_fill_report()` — in this
NautilusTrader version those reports live on `engine.trader`:

| Report | Method |
|---|---|
| Order fill report | `engine.trader.generate_order_fills_report()` |
| Position report   | `engine.trader.generate_positions_report()` |
| Account report    | `engine.trader.generate_account_report(venue)` |

In [9]:
venue_obj = Venue(VENUE)

order_fill_report = engine.trader.generate_order_fills_report()
position_report = engine.trader.generate_positions_report()
account_report = engine.trader.generate_account_report(venue_obj)

print(f"Order fill report : {len(order_fill_report):,} rows")
print(f"Position report   : {len(position_report):,} rows")
print(f"Account report    : {len(account_report):,} rows")

if len(order_fill_report) == 0:
    print("\nWARNING: no fills — check that both ASK and BID data loaded.")

# Persist alongside the notebook.
out_dir = Path.cwd()
order_fill_report.to_csv(out_dir / "ema_cross_order_fill_report.csv")
position_report.to_csv(out_dir / "ema_cross_position_report.csv")
account_report.to_csv(out_dir / "ema_cross_account_report.csv")
print(f"\nReports saved as CSV in: {out_dir}")

Order fill report : 174 rows
Position report   : 87 rows
Account report    : 349 rows

Reports saved as CSV in: d:\mcube_test_version_1_updated\m-cube_version1\strategies


### 8a. Order fill report

In [10]:
order_fill_report

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20200401-044500-001-000-1,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-001,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-002,MARKET,BUY,100000,...,None,None,None,None,None,None,[EMA Cross BUY: fast(10)=20.50533 >= slow(20)=...,f5f0ba83-bd58-4162-bcdc-b321dd6ff9ce,2020-04-01 04:45:00+00:00,2020-04-01 04:45:00+00:00
O-20200401-054500-001-000-2,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-002,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-004,MARKET,SELL,100000,...,None,None,None,None,None,None,None,45813312-5db1-46ea-814f-7ede63da9b5c,2020-04-01 05:45:00+00:00,2020-04-01 05:45:00+00:00
O-20200401-054500-001-000-3,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-003,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-006,MARKET,SELL,100000,...,None,None,None,None,None,None,[EMA Cross SELL: fast(10)=20.44694 < slow(20)=...,34096a38-ad36-4849-860a-4f3ee6dd7153,2020-04-01 05:45:00+00:00,2020-04-01 05:45:00+00:00
O-20200401-103000-001-000-4,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-004,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-008,MARKET,BUY,100000,...,None,None,None,None,None,None,None,d13d94fa-9338-4d1f-bfba-6fa5b0ae8d65,2020-04-01 10:30:00+00:00,2020-04-01 10:30:00+00:00
O-20200401-103000-001-000-5,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-005,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-010,MARKET,BUY,100000,...,None,None,None,None,None,None,[EMA Cross BUY: fast(10)=20.29533 >= slow(20)=...,b491ad5c-4433-4a10-847e-a633e1ddd113,2020-04-01 10:30:00+00:00,2020-04-01 10:30:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O-20200430-174500-001-000-170,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-170,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-340,MARKET,SELL,100000,...,None,None,None,None,None,None,None,4f3d44e6-703a-4aa6-8c62-3f1e948e4953,2020-04-30 17:45:00+00:00,2020-04-30 17:45:00+00:00
O-20200430-174500-001-000-171,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-171,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-342,MARKET,SELL,100000,...,None,None,None,None,None,None,[EMA Cross SELL: fast(10)=17.55293 < slow(20)=...,5734a888-99f2-4d85-afb2-204e222d06b0,2020-04-30 17:45:00+00:00,2020-04-30 17:45:00+00:00
O-20200430-183000-001-000-172,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-1-172,EURUSD.Commodities_MS-EMACrossTFStrategy-000,Commodities_MS-001,Commodities_MS-1-344,MARKET,BUY,100000,...,None,None,None,None,None,None,None,a5964e83-7d55-44ff-83c2-ed3dca7034cb,2020-04-30 18:30:00+00:00,2020-04-30 18:30:00+00:00


### 8b. Position report

In [11]:
position_report

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
EURUSD.Commodities_MS-EMACrossTFStrategy-000-86eb76b6-617c-46e4-813d-5a206a6ea25f,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200401-044500-001-000-1,O-20200401-054500-001-000-2,BUY,FLAT,0,100000,...,2020-04-01 04:45:00+00:00,1585719900000000000,2020-04-01 05:45:00+00:00,3600000000000,20.53501,20.26499,[4080.00 USD],-0.01315,-31082.00 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-2c02f9d9-6d6d-4fde-95f3-e872fa1cacd5,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200401-054500-001-000-3,O-20200401-103000-001-000-4,SELL,FLAT,0,100000,...,2020-04-01 05:45:00+00:00,1585737000000000000,2020-04-01 10:30:00+00:00,17100000000000,20.26499,20.56501,[4083.00 USD],-0.01480,-34085.00 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-d9552041-8317-49d0-bcde-026d9b002f16,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200401-103000-001-000-5,O-20200401-130000-001-000-6,BUY,FLAT,0,100000,...,2020-04-01 10:30:00+00:00,1585746000000000000,2020-04-01 13:00:00+00:00,9000000000000,20.56501,20.25499,[4082.00 USD],-0.01508,-35084.00 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-ef71edd7-348c-4087-80d5-3d2c7e10b591,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200401-130000-001-000-7,O-20200401-131500-001-000-8,SELL,FLAT,0,100000,...,2020-04-01 13:00:00+00:00,1585746900000000000,2020-04-01 13:15:00+00:00,900000000000,20.25499,20.51501,[4077.00 USD],-0.01284,-30079.00 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-1b4ad488-2f14-45c5-88bf-8c8d3a9bdee5,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200401-131500-001-000-9,O-20200401-134500-001-000-10,BUY,FLAT,0,100000,...,2020-04-01 13:15:00+00:00,1585748700000000000,2020-04-01 13:45:00+00:00,1800000000000,20.51501,20.32499,[4084.00 USD],-0.00926,-23086.00 USD,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EURUSD.Commodities_MS-EMACrossTFStrategy-000-6030e75a-d801-48f2-8862-b3e830053166,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200429-220000-001-000-165,O-20200430-131500-001-000-166,BUY,FLAT,0,100000,...,2020-04-29 22:00:00+00:00,1588252500000000000,2020-04-30 13:15:00+00:00,54900000000000,15.58501,16.83999,[3242.51 USD],0.08052,122255.50 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-24e0cf7c-048f-42af-ae1a-6705cc262049,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200430-131500-001-000-167,O-20200430-140000-001-000-168,SELL,FLAT,0,100000,...,2020-04-30 13:15:00+00:00,1588255200000000000,2020-04-30 14:00:00+00:00,2700000000000,16.83999,17.98001,[3482.00 USD],-0.06770,-117484.00 USD,True
EURUSD.Commodities_MS-EMACrossTFStrategy-000-04e4b915-cfc2-4b18-9503-d8a565717c8e,BACKTESTER-001,EMACrossTFStrategy-000,EURUSD.Commodities_MS,Commodities_MS-001,O-20200430-140000-001-000-169,O-20200430-174500-001-000-170,BUY,FLAT,0,100000,...,2020-04-30 14:00:00+00:00,1588268700000000000,2020-04-30 17:45:00+00:00,13500000000000,17.98001,17.23999,[3522.00 USD],-0.04116,-77524.00 USD,True


### 8c. Account report

In [ ]:
account_report

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2020-04-01 00:00:00+00:00,1000000.00,0.00,1000000.00,USD,Commodities_MS-001,MARGIN,USD,[],True,{}
2020-04-01 04:45:00+00:00,999999.98,7.19,999992.79,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-04-01 04:45:00+00:00,997946.50,718725.35,279221.15,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-04-01 05:45:00+00:00,997946.21,718718.16,279228.05,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-04-01 05:45:00+00:00,968918.00,0.00,968918.00,USD,Commodities_MS-001,MARGIN,USD,[],False,{}
...,...,...,...,...,...,...,...,...,...,...
2020-04-30 18:30:00+00:00,1339685.70,0.00,1339685.70,USD,Commodities_MS-001,MARGIN,USD,[],False,{}
2020-04-30 18:30:00+00:00,1339685.68,6.47,1339679.21,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-04-30 18:30:00+00:00,1337837.70,646800.35,691037.35,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-04-30 23:59:00+00:00,1337838.74,646793.88,691044.86,USD,Commodities_MS-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}


: 

## 9. Summary

In [13]:
accounts = list(engine.kernel.cache.accounts())
final_balance = STARTING_CAPITAL
if accounts:
    bal = accounts[0].balance_total(USD)
    if bal is not None:
        final_balance = float(bal)

pnl = final_balance - STARTING_CAPITAL
positions = engine.kernel.cache.positions()
closed = [p for p in positions if p.is_closed]

print(f"Timeframe         : {TIMEFRAME} ({AGG_SPEC})")
print(f"Composite bar type: {instrument.id}-{AGG_SPEC}-BID-INTERNAL@1-MINUTE-EXTERNAL")
print(f"Order fills       : {len(order_fill_report):,}")
print(f"Closed positions  : {len(closed):,}")
print(f"Starting capital  : {STARTING_CAPITAL:,.2f} USD")
print(f"Final balance     : {final_balance:,.2f} USD")
print(f"Total P&L         : {pnl:,.2f} USD ({pnl / STARTING_CAPITAL * 100:.2f}%)")

engine.dispose()

Timeframe         : 15min (15-MINUTE)
Composite bar type: EURUSD.FOREX_MS-15-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL
Order fills       : 164
Closed positions  : 1
Starting capital  : 1,000,000.00 USD
Final balance     : 981,530.89 USD
Total P&L         : -18,469.11 USD (-1.85%)
